# CORD receipts: zero-shot VLM probe (Qwen2.5-VL-3B)

**Purpose.** Before deciding whether to switch the capstone to a vision-language (VLM) project,
measure how well a small VLM reads receipts *without* fine-tuning. If the zero-shot score is already
very high, fine-tuning has little to show; if it is low or middling, CORD is a good fine-tuning
target.

- **Data:** [CORD v2](https://huggingface.co/datasets/naver-clova-ix/cord-v2), Indonesian
  restaurant receipts, CC-BY-4.0. 800 train / 100 validation / 100 test photos, each labelled with a
  JSON parse (`menu` items, `sub_total`, `total`). This notebook uses **validation only**; test stays
  closed.
- **Model:** `Qwen/Qwen2.5-VL-3B-Instruct`, greedy decoding, no fine-tuning.
- **Score:** field-level F1 as in the Donut paper (every `section.key = value` pair, exact string
  match, micro-averaged), with a bootstrap 95% range; a lenient F1 (ignores case and spaces);
  exact `total_price` accuracy; valid-JSON rate.

**Run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. About 20 to 40
minutes for 100 receipts. Predictions are saved after every receipt, so a rerun resumes.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "transformers>=4.49" accelerate datasets
import os, subprocess, sys
REPO = "https://github.com/parulrewatkar0692/cord-receipt-vlm"
if not os.path.exists("/content/cord-receipt-vlm"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/cord-receipt-vlm"], check=True)
sys.path.insert(0, "/content/cord-receipt-vlm/eval")
from score import *   # parse, fields, counts, doc_scores, f1, score, overlap_ids, load_labels

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DATASET = "naver-clova-ix/cord-v2"
SPLIT = "validation"          # never "test" while exploring
LIMIT = None                  # e.g. 10 for a quick try
MAX_NEW_TOKENS = 1024
MAX_PIXELS = 1024 * 28 * 28   # caps the image size so a T4 does not run out of memory
PRED_PATH = f"/content/cord_{SPLIT}_qwen25vl3b_zeroshot.jsonl"

## 2. Data

In [ ]:
import json
from datasets import load_dataset

ds = load_dataset(DATASET, split=SPLIT)
rows = []
for i, ex in enumerate(ds):
    gt = json.loads(ex["ground_truth"])
    rows.append({"id": gt["meta"]["image_id"], "image": ex["image"], "gold": gt["gt_parse"]})
if LIMIT:
    rows = rows[:LIMIT]
print(len(rows), "receipts")
print(json.dumps(rows[0]["gold"], indent=1)[:800])
rows[0]["image"].resize((rows[0]["image"].width // 3, rows[0]["image"].height // 3))

**Overlap with train.** Some receipts have exactly the same label as a train receipt (the same
receipt or order photographed again: 14 of 100 in validation). They are scored with the rest and
again without them, so a later fine-tuned model cannot look good by memorising train.

In [ ]:
train_labels = [r["gold"] for r in load_labels("train")]   # label column only, no images
overlap = overlap_ids(rows, train_labels)
print(f"{len(overlap)} of {len(rows)} {SPLIT} receipts have a label identical to a train receipt")

## 3. Prompt

The prompt lists the CORD field names so the model can answer in the same schema. The check
below fails if the labels use a field the prompt does not mention.

In [ ]:
PROMPT = """Read this receipt and return its contents as one JSON object, and nothing else.
Copy every value exactly as printed on the receipt: same spelling, digits, separators and symbols.
Leave out any field that is not on the receipt. Use only these keys:
- "menu": list of purchased items; each item may have "nm" (item name), "num" (item code),
  "unitprice", "cnt" (quantity), "discountprice", "price" (line amount), "itemsubtotal",
  "vatyn", "etc", and "sub" (list of add-ons or modifiers, each with "nm", "unitprice", "cnt",
  "price", "etc")
- "void_menu": list of cancelled items, each with "nm" and "price"
- "sub_total": "subtotal_price", "discount_price", "service_price", "othersvc_price",
  "tax_price", "etc"
- "total": "total_price", "total_etc", "cashprice", "changeprice", "creditcardprice",
  "emoneyprice", "menutype_cnt" (number of item types), "menuqty_cnt" (number of items)"""

In [ ]:
prompt_keys = set(re.findall(r'"([a-z_]+)"', PROMPT))
label_keys = {part for g in rows for path, _ in fields(g["gold"]) for part in path.split(".")}
missing = label_keys - prompt_keys
assert not missing, f"labels use keys the prompt does not list: {missing}"
print("prompt covers all", len(label_keys), "label keys")

## 4. Load the model

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
# T4 has no native bfloat16; float16 halves memory (3B model is about 7.5 GB)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
if "model" not in globals():
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map="cuda", attn_implementation="sdpa")
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256 * 28 * 28,
                                              max_pixels=MAX_PIXELS)
model.eval()
print(MODEL_ID, dtype, f"{torch.cuda.memory_allocated() / 2**30:.1f} GiB used")

## 5. Run on the validation receipts

One receipt at a time, greedy. Results are appended to `PRED_PATH`, so rerunning this cell
continues where it stopped. On an out-of-memory error the image is halved and tried again.

In [ ]:
import os, time

def generate(image):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]

done = {}
if os.path.exists(PRED_PATH):
    with open(PRED_PATH) as f:
        done = {p["id"]: p for p in map(json.loads, f)}
print(f"{len(done)} already done")

start = time.time()
with open(PRED_PATH, "a") as f:
    for k, r in enumerate(rows):
        if r["id"] in done:
            continue
        image = r["image"].convert("RGB")
        t0 = time.time()
        try:
            output = generate(image)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            output = generate(image.resize((image.width // 2, image.height // 2)))
        done[r["id"]] = {"id": r["id"], "output": output, "seconds": round(time.time() - t0, 1)}
        f.write(json.dumps(done[r["id"]], ensure_ascii=False) + "\n")
        f.flush()
        if (k + 1) % 10 == 0 or k + 1 == len(rows):
            print(f"{k + 1}/{len(rows)} receipts, {time.time() - start:.0f}s")
print(done[rows[0]["id"]]["output"][:600])

## 6. Score

In [ ]:
golds = [r["gold"] for r in rows]
outputs = [done[r["id"]]["output"] for r in rows]
report, per_doc = score(golds, outputs)
keep = [i for i, r in enumerate(rows) if r["id"] not in overlap]
report_no_overlap, _ = score([golds[i] for i in keep], [outputs[i] for i in keep])
seconds = round(sum(done[r["id"]]["seconds"] for r in rows) / len(rows), 1)

def show(name, a, b):
    fmt = lambda v: "n/a" if v is None else f"{v:.3f}"
    print(f"{name:26s} {fmt(a):>10s} {fmt(b):>14s}")

print(f"{'metric':26s} {'all (n=' + str(report['n']) + ')':>10s} "
      f"{'no overlap (n=' + str(report_no_overlap['n']) + ')':>14s}")
for k in ["json_valid", "json_parsed", "field_f1", "field_f1_lenient", "total_price_acc"]:
    show(k, report[k], report_no_overlap[k])
for k in ["field_f1_ci95"]:
    print(f"{k:26s} {report[k][0]:.3f}-{report[k][1]:.3f}    "
          f"{report_no_overlap[k][0]:.3f}-{report_no_overlap[k][1]:.3f}")
for sec in report["by_section"]:
    show("  f1 " + sec, report["by_section"][sec], report_no_overlap["by_section"][sec])
print(f"seconds per receipt: {seconds}")

**Reading the score.** Published fine-tuned results on CORD are roughly 0.84 field F1
(Donut, OCR-free) to about 0.96 (LayoutLMv3, uses OCR). A zero-shot score well below that leaves
clear room for fine-tuning to show a gain.

## 7. Where it goes wrong: the 3 lowest-scoring receipts

In [ ]:
worst = sorted(range(len(rows)), key=lambda i: f1(*per_doc[i]["all"]) or 0)[:3]
for i in worst:
    g, p = counts(golds[i]), counts(parse(outputs[i])[0] or {})
    print(f"--- receipt {rows[i]['id']}: F1 {f1(*per_doc[i]['all']) or 0:.2f}")
    print("  missed:", sorted((g - p).elements())[:8])
    print("  wrong or extra:", sorted((p - g).elements())[:8])

## 8. Save and download

In [ ]:
REPORT_PATH = PRED_PATH.replace(".jsonl", "_report.json")
full = {"model": MODEL_ID, "split": SPLIT, "max_pixels": MAX_PIXELS, "seconds_per_receipt": seconds,
        "all": report, "without_train_overlap": report_no_overlap, "overlap_ids": sorted(overlap)}
with open(REPORT_PATH, "w") as f:
    json.dump(full, f, indent=2)
print(json.dumps(full, indent=2))
try:
    from google.colab import files
    files.download(PRED_PATH)
    files.download(REPORT_PATH)
except ImportError:
    print("saved to", PRED_PATH, "and", REPORT_PATH)